In [16]:
import numpy as np
import pandas as pd

In [17]:
file_path = "C:/Users/sahit/Downloads/priceoye_laptops_version_2.csv"
data = pd.read_csv(file_path)

In [18]:
print(data.isnull().sum())

Unnamed: 0            0
Name                  0
Discounted Price      0
Actual Price         15
Saving               15
Rating              280
Reviews             280
Brand                 0
Core                 87
SSD                 214
Model                 0
dtype: int64


In [21]:
data.drop(columns=['Unnamed: 0', 'Rating', 'SSD', 'Reviews', 'Model'],inplace=True)


In [22]:
data.isnull().sum()

Name                 0
Discounted Price     0
Actual Price        15
Saving              15
Brand                0
Core                87
dtype: int64

In [23]:
data['Brand'] = data['Brand'].str.lower()

In [24]:
data['Core'] = data['Core'].fillna(
   data['Name'].str.extract('(i3|i5|i7|i9|Ryzen 3|Ryzen 5|Ryzen 7|M1|M4|Ryzen 9)', expand=False)
) 

In [25]:
# convert to lowercase first
data['Core'] = data['Core'].str.lower()

# extract only processor family properly
data['Core'] = data['Core'].str.extract(
    r'(i3|i5|i7|i9|'
    r'ryzen\s?\d|'
    r'ultra\s?\d|'
    r'm\d)',
    expand=False
)

In [26]:
brand_plus_core = data.groupby(['Brand', 'Core'])['Actual Price'].median()
print(brand_plus_core)

Brand    Core   
acer     i5          275000.0
         i7          305000.0
         i9          474999.0
         ultra 9     724000.0
apple    i5          380000.0
         m1          258000.0
         m2          410000.0
         m3          449999.0
         m4          324999.5
asus     i5          200000.0
         i7          365000.0
         m1          215000.0
         ryzen 9     550000.0
         ultra 5     250000.0
         ultra 7     315000.0
         ultra 9     549999.5
dell     i3          139999.0
         i5          200000.0
         i7          304999.0
         ryzen 5     145000.0
         ryzen 7     142999.0
         ultra 5     295000.0
         ultra 7     374999.0
         ultra 9     750000.0
hp       i3          125000.0
         i5          200000.0
         i7          310000.0
         m0          280000.0
         ryzen 5     199999.0
         ryzen 7     247499.5
         ultra 5     262500.0
         ultra 7     340000.0
         ultra 9     37

In [27]:
data['Actual Price'] = data['Actual Price'].fillna(
    data.groupby(['Brand','Core'])['Actual Price'].transform('median'))

In [28]:
data.isnull().sum()

Name                 0
Discounted Price     0
Actual Price         0
Saving              15
Brand                0
Core                34
dtype: int64

In [29]:
data['Saving'] = (((data['Actual Price'] - data['Discounted Price'])/data['Actual Price'])*100)

In [30]:
data.isnull().sum()

Name                 0
Discounted Price     0
Actual Price         0
Saving               0
Brand                0
Core                34
dtype: int64

In [31]:
import numpy as np

# Calculate group-wise Z-score
data['Z_score'] = data.groupby(['Brand', 'Core'])['Actual Price'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Set threshold
threshold = 3

# Identify outliers
outliers = data[np.abs(data['Z_score']) > threshold]

print("Total outliers:", outliers.shape)
outliers.head()

Total outliers: (1, 7)


,Name,Discounted Price,Actual Price,Saving,Brand,Core,Z_score
48,HP Envy 16 H1008TX 16 Inches 13th Gen Core i5 ...,409999.0,449999.0,8.888909,hp,i5,3.676595


In [32]:
data.drop(columns=['Core'],inplace=True)


In [33]:
data.columns

Index(['Name', 'Discounted Price', 'Actual Price', 'Saving', 'Brand',
       'Z_score'],
      dtype='object')

In [34]:
data.to_csv('price_of_laptops_preprocessed.csv',index=False)